<a href="https://colab.research.google.com/github/diademsamuel/7316/blob/main/SOIL_DOWNSCALINGipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade xee

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.5/477.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.0 MB/s eta 0:00:00
  Attempting uninstall: earthengine-api
    Found existing installation: earthengine-api 1.5.24
    Uninstalling earthengine-api-1.5.24:
      Successfully uninstalled earthengine-api-1.5.24


In [ ]:
!pip install --upgrade geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(
    project='ee-diademsamuel',
    opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map= geemap.Map
map

geemap.geemap.Map

In [ ]:
m = geemap.Map()
m.add("basemap_selector")
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [ ]:
m = geemap.Map(center=[33.5, -101.5], zoom=6)

# Helper function
def poly(coords):
    return ee.Geometry.Polygon(coords)
    # High Plains
high_plains = poly([
    [
        [-103.04, 36.5],
        [-100.00, 36.5],
        [-100.00, 32.5],
        [-103.04, 32.5]
    ]
])

# Hill Country
hill_country = poly([
    [
        [-100.5, 31.5],
        [-98.0, 31.5],
        [-97.5, 30.0],
        [-98.5, 29.5],
        [-100.0, 29.8],
        [-100.5, 31.0]
    ]
])

In [ ]:
m.addLayer(high_plains, {'color': 'blue'}, 'High Plains')
m.addLayer(hill_country, {'color': 'green'}, 'Hill Country')

m


Map(center=[33.5, -101.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [ ]:
roi

In [ ]:
roi = high_plains
# Alternatively, you could use:
# roi = hill_country

print(roi)

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              -103.04,
              36.5
            ],
            [
              -100.0,
              36.5
            ],
            [
              -100.0,
              32.5
            ],
            [
              -103.04,
              32.5
            ]
          ]
        ]
      },
      "evenOdd": {
        "constantValue": true
      }
    }
  }
})


In [ ]:
# Combine high_plains and hill_country into a single ROI
roi_combined = high_plains.union(hill_country)

# Assign the combined geometry to roi
roi = roi_combined

print(roi)

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Geometry.union",
    "arguments": {
      "left": {
        "functionInvocationValue": {
          "functionName": "GeometryConstructors.Polygon",
          "arguments": {
            "coordinates": {
              "constantValue": [
                [
                  [
                    -103.04,
                    36.5
                  ],
                  [
                    -100.0,
                    36.5
                  ],
                  [
                    -100.0,
                    32.5
                  ],
                  [
                    -103.04,
                    32.5
                  ]
                ]
              ]
            },
            "evenOdd": {
              "constantValue": true
            }
          }
        }
      },
      "right": {
        "functionInvocationValue": {
          "functionName": "GeometryConstructors.Polygon",
          "arguments": {
            "

In [ ]:
# Optionally, add the combined ROI to the map to visualize it
m.addLayer(roi, {'color': 'red'}, 'Combined ROI')
m

Map(bottom=6872.0, center=[33.5, -101.5], controls=(WidgetControl(options=['position', 'transparent_bg'], posi…

In [ ]:
roi_high_plains = high_plains
roi_hill_country = hill_country

print('ROI for High Plains:', roi_high_plains)
print('ROI for Hill Country:', roi_hill_country)

ROI for High Plains: ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              -103.04,
              36.5
            ],
            [
              -100.0,
              36.5
            ],
            [
              -100.0,
              32.5
            ],
            [
              -103.04,
              32.5
            ]
          ]
        ]
      },
      "evenOdd": {
        "constantValue": true
      }
    }
  }
})
ROI for Hill Country: ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              -100.5,
              31.5
            ],
            [
              -98.0,
              31.5
            ],
            [
              -97.5,
              30.0
            ],
           

In [ ]:
time_start= ee.Date('2015')
time_end  = ee.Date('2026')
time_dif = time_end.difference(time_start, 'month').round()
time_list = ee.List.sequence(0, ee.Number(time_dif).subtract(1)).map(
    lambda x: time_start.advance(x, 'month')
)

smap = (
    ee.ImageCollection("NASA/SMAP/SPL3SMP_E/005")
           .filterDate(time_start, time_end)
           .select(['soil_moisture_am'],['sm'])
)

# Modified monthly function to ensure consistent bands
def monthly (date, col, output_bands):
    start_date = ee.Date(date)
    end_date = start_date.advance(1, 'month')
    filtered_col = col.filterDate(start_date, end_date)
    col_agg = filtered_col.mean()

    # Create a fully masked image with the expected output_bands as a template
    # Create as many constant bands as there are output_bands
    template_image = ee.Image.constant([0] * len(output_bands)).rename(output_bands).updateMask(ee.Image.constant(0))

    # Select bands from col_agg if it has bands, otherwise use a masked template
    selected_col_agg = ee.Algorithms.If(
        col_agg.bandNames().size().gt(0),
        col_agg.select(output_bands),
        template_image
    )
    # Blend the selected aggregated image with the template to ensure all bands are present
    final_image = template_image.blend(ee.Image(selected_col_agg))

    return final_image.set('system:time_start', start_date.millis())

smap_monthly = ee.ImageCollection(
    time_list.map(lambda x: monthly(x, smap, ['sm']))
)

smap_monthly_mean = smap_monthly.mean()
smap_monthly_mean
ndvi = (
    ee.ImageCollection("MODIS/061/MOD13Q1")
    .filterDate(time_start, time_end)
    .select(['NDVI','EVI'],['ndvi','evi'])
)

ndvi_monthly = ee.ImageCollection(
    time_list.map(
        lambda x: monthly(x, ndvi, ['ndvi','evi'])
    )
)

temp = (
    ee.ImageCollection("MODIS/061/MOD11A2")
    .filterDate(time_start, time_end)
    .select(['LST_Day_1km','LST_Night_1km'],['temp_day','temp_night'])
)

temp_monthly = ee.ImageCollection(
    time_list.map(
        lambda x: monthly(x, temp, ['temp_day','temp_night'])
    )
)

et = (
    ee.ImageCollection("MODIS/061/MOD16A2GF")
    .filterDate(time_start, time_end)
    .select(['ET'],['et'])
)

et_monthly = ee.ImageCollection(
    time_list.map(
        lambda x: monthly(x, et, ['et'])
    )
)

landcover = (
    ee.ImageCollection("MODIS/061/MCD12Q1")
    .filterDate(time_start, time_end)
    .select('LC_Type1')
    .mode().rename('landcover')
)

topo = ee.Image("USGS/GTOPO30")

# New approach to combine collections, ensuring all bands are explicitly merged per month
def combine_all_monthly_data(date):
    # Get monthly aggregated images from each collection
    # Now using the modified 'monthly' function which ensures bands are always present
    smap_img = monthly(date, smap, ['sm']) # Returns image with 'sm' band (or masked 'sm')
    ndvi_img = monthly(date, ndvi, ['ndvi', 'evi']) # Returns image with 'ndvi', 'evi' bands
    temp_img = monthly(date, temp, ['temp_day', 'temp_night']) # Returns image with 'temp_day', 'temp_night' bands
    et_img = monthly(date, et, ['et'])     # Returns image with 'et' band

    # Combine all these images into a single image for the month
    # Using addBands should work correctly now that each monthly image is guaranteed to have its bands.
    combined_img = smap_img.addBands(ndvi_img).addBands(temp_img).addBands(et_img)

    # Add static bands (topo and landcover)
    final_img = combined_img.addBands(topo).addBands(landcover)
    return final_img

# Create the collection using the new combine function
collection = ee.ImageCollection(time_list.map(combine_all_monthly_data))

collection

import xarray as xr
import xee

ds10km = xr.open_dataset(
    collection,
    engine ='ee',
    crs = 'EPSG:4326',
    scale = 0.1,
    geometry = roi
)
ds10km = ds10km.sortby('time') * 1

df10km = ds10km.to_dataframe().dropna()

df10km

sm    ndvi     evi      temp_day  \
time       lon     lat                                                 
2015-03-01 -102.99 29.55  1.325400e+09  2487.5  1579.0  15034.750000   
                   29.65  1.325400e+09  2532.5  1543.0  15107.500000   
                   29.75  1.325400e+09  2111.0  1652.0  14991.250000   
                   29.85  1.325400e+09  2295.5  1696.0  15066.666992   
                   29.95  1.325400e+09  2219.0  1299.5  14981.000000   
...                                ...     ...     ...           ...   
2023-12-01 -97.59  30.05  2.399321e-01  6050.0  3187.5  14469.750000   
                   30.15  1.991472e-01  5611.5  3071.0  14477.500000   
                   30.25  1.815287e-01  5110.5  2843.5  14432.500000   
                   30.35  1.949408e-01  4232.5  2331.0  14483.500000   
                   30.45  2.175649e-01  4154.5  2238.0  14536.500000   

                          temp_night     et  elevation  landcover  
time       lon     lat                                             
2015-03-01 -102.99 29.55    14292.25  55.50      869.0        7.0  
                   29.65    14307.50  56.00      802.0        7.0  
                   29.75    14257.75  50.75      835.0        7.0  
                   29.85    14220.25  52.00      902.0        7.0  
                   29.95    14211.75  50.50     1077.0       10.0  
...                              ...    ...        ...        ...  
2023-12-01 -97.59  30.05    13996.00  63.50      159.0       10.0  
                   30.15    14005.75  59.75      146.0       10.0  
                   30.25    14020.25  56.25      134.0       10.0  
                   30.35    14018.00  50.50      172.0       10.0  
                   30.45    13961.25  50.00      190.0       10.0  

[405420 rows x 8 columns]

In [ ]:
from sklearn.preprocessing import StandardScaler

x10km = df10km[['ndvi','evi', 'et','temp_day','temp_night', 'elevation', 'landcover']]
scaler_x10km = StandardScaler()
x10km_standard = scaler_x10km.fit_transform(x10km)
x10km_standard

array([[-1.1830599 , -0.9989154 , -0.42774445, ..., -0.0193139 ,
         0.7497875 , -2.472356  ],
       [-1.1452862 , -1.0463079 , -0.41732275, ...,  0.0156838 ,
         0.50581867, -2.472356  ],
       [-1.4991007 , -0.90281385, -0.52675056, ..., -0.09848902,
         0.6259824 , -2.472356  ],
       ...,
       [ 1.0187321 ,  0.66574734, -0.4121119 , ..., -0.6435351 ,
        -1.9265875 ,  0.07608487],
       [ 0.28172356, -0.00893802, -0.5319614 , ..., -0.6486987 ,
        -1.788217  ,  0.07608487],
       [ 0.21624899, -0.13136873, -0.5423831 , ..., -0.77893597,
        -1.722673  ,  0.07608487]], dtype=float32)

In [ ]:
y10km = df10km['sm']
scaler_y10km = StandardScaler()
y10km_standard = scaler_y10km.fit_transform(y10km.values.reshape(-1,1))
y10km_standard

array([[ 1.6644597 ],
       [ 1.6644597 ],
       [ 1.6644597 ],
       ...,
       [-0.87939847],
       [-0.87939847],
       [-0.87939847]], dtype=float32)

In [ ]:
# Print the ds10km object to see its data variables (bands) and coordinates
print(ds10km)

<xarray.Dataset> Size: 16MB
Dimensions:     (time: 132, lon: 55, lat: 70)
Coordinates:
  * time        (time) datetime64[ns] 1kB 2015-01-01 2015-02-01 ... 2025-12-01
  * lon         (lon) float64 440B -103.0 -102.9 -102.8 ... -97.79 -97.69 -97.59
  * lat         (lat) float64 560B 29.55 29.65 29.75 29.85 ... 36.25 36.35 36.45
Data variables:
    sm          (time, lon, lat) float32 2MB nan nan nan nan ... nan nan nan nan
    ndvi        (time, lon, lat) float32 2MB 2.37e+03 2.162e+03 ... 3.702e+03
    evi         (time, lon, lat) float32 2MB 1.161e+03 1.054e+03 ... 2.16e+03
    temp_day    (time, lon, lat) float32 2MB 1.445e+04 1.454e+04 ... 1.407e+04
    temp_night  (time, lon, lat) float32 2MB 1.393e+04 1.393e+04 ... 1.38e+04
    et          (time, lon, lat) float32 2MB 68.0 68.25 62.25 ... 48.0 49.0
    elevation   (time, lon, lat) float32 2MB 869.0 802.0 835.0 ... 337.0 344.0
    landcover   (time, lon, lat) float32 2MB 7.0 7.0 7.0 7.0 ... 10.0 12.0 12.0
Attributes:
    crs:      E

In [ ]:

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x10km_standard, y10km_standard, test_size = 0.2, random_state = 42
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
x_train.shape[1]


7

In [ ]:
model = Sequential()
model.add(Dense(64, activation = 'relu', input_dim = x_train.shape[1]))
model.add(Dense(32, activation = 'relu'))
model.add(Dense(16, activation = 'relu'))
model.add(Dense(1))
model.compile(
    optimizer = 'adam',
    loss = 'mse',
    metrics = ['mae']
)
model.fit(
    x_train, y_train,
    epochs = 100,
    batch_size = 8,
    validation_split = 0.2,
    verbose = 1
)

Epoch 1/100
32434/32434 ━━━━━━━━━━━━━━━━━━━━ 91s 3ms/step - loss: 0.9166 - mae: 0.9108 - val_loss: 0.8798 - val_mae: 0.8876
Epoch 2/100
32434/32434 ━━━━━━━━━━━━━━━━━━━━ 142s 3ms/step - loss: 0.8493 - mae: 0.8488 - val_loss: 0.8378 - val_mae: 0.8457
Epoch 3/100
32434/32434 ━━━━━━━━━━━━━━━━━━━━ 96s 3ms/step - loss: 0.8170 - mae: 0.8196 - val_loss: 0.8136 - val_mae: 0.8210
Epoch 4/100
32434/32434 ━━━━━━━━━━━━━━━━━━━━ 90s 3ms/step - loss: 0.7985 - mae: 0.8035 - val_loss: 0.8081 - val_mae: 0.8055
Epoch 5/100
32434/32434 ━━━━━━━━━━━━━━━━━━━━ 92s 3ms/step - loss: 0.7858 - mae: 0.7924 - val_loss: 0.7858 - val_mae: 0.7962
Epoch 6/100
32434/32434 ━━━━━━━━━━━━━━━━━━━━ 140s 3ms/step - loss: 0.7752 - mae: 0.7828 - val_loss: 0.7808 - val_mae: 0.8006
Epoch 7/100
32434/32434 ━━━━━━━━━━━━━━━━━━━━ 89s 3ms/step - loss: 0.7672 - mae: 0.7754 - val_loss: 0.7751 - val_mae: 0.7665
Epoch 8/100
32434/32434 ━━━━━━━━━━━━━━━━━━━━ 89s 3ms/step - loss: 0.7605 - mae: 0.7695 - val_loss: 0.7594 - val_mae: 0.7666
Epoch 

In [ ]:
loss, mae = model.evaluate(x_test, y_test)

y_pred = model.predict(x_test)

In [ ]:

from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
import numpy as np

In [ ]:
r2 = r2_score(y_test, y_pred)
print(f'R2: {r2}')
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'RMSE: {rmse}')

NameError: name 'y_test' is not defined

In [ ]:
ds1km = xr.open_dataset(
    collection,
    engine = 'ee',
    crs = 'EPSG:4326',
    scale = 0.01,
    geometry = roi
)

In [ ]:
ds1km = ds1km.sortby('time') * 1

df1km = ds1km.to_dataframe().dropna()


In [ ]:
x1km = df1km[['ndvi','evi', 'et','temp_day','temp_night', 'elevation', 'landcover']]
scaler_x1km = StandardScaler()
x1km_standard = scaler_x1km.fit_transform(x1km)
x1km_standard


array([[-1.2144432 , -1.1153482 , -0.46239984, ..., -0.12169053,
         1.6866739 , -2.4335277 ],
       [-1.2711    , -1.0904112 , -0.47236264, ..., -0.09359355,
         1.7661724 , -2.4335277 ],
       [-1.166532  , -0.990083  , -0.45243704, ..., -0.08327222,
         1.5710397 , -2.4335277 ],
       ...,
       [ 1.3754231 ,  1.2339541 , -0.4125859 , ..., -0.79143053,
        -1.7317612 ,  0.05999702],
       [ 0.89973336,  0.59718937, -0.51221377, ..., -0.7942976 ,
        -1.7606697 ,  0.05999702],
       [ 0.8001085 ,  0.44350752, -0.50723237, ..., -0.869414  ,
        -1.7353748 ,  0.05999702]], dtype=float32)

In [ ]:
y1km = df1km['sm']
scaler_y1km = StandardScaler()
y1km_standard = scaler_y1km.fit_transform(y1km.values.reshape(-1,1))
y1km_standard

NameError: name 'df1km' is not defined

In [ ]:
df1km['sm_1km'] = scaler_y1km.inverse_transform(model.predict(x1km_standard))
